In [1]:
# # Install Pytorch & other libraries
# %pip install "torch==2.4.1" tensorboard 
# %pip install flash-attn "setuptools<71.0.0" scikit-learn 
 
# # Install Hugging Face libraries
# %pip install  --upgrade \
#   "datasets==3.1.0" \
#   "accelerate==1.2.1" \
#   "hf-transfer==0.1.8"
#   #"transformers==4.47.1" \
 
# # ModernBERT is not yet available in an official release, so we need to install it from github
# %pip install "git+https://github.com/huggingface/transformers.git@6e0515e99c39444caae39472ee1b2fd76ece32f1" --upgrade

In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
from datasets import load_dataset
 
# Dataset id from huggingface.co/dataset
dataset_id = "youralien/feedback_qesconv_16wayclassification"
 
# Load raw dataset
raw_dataset = load_dataset(dataset_id, split="train") # happens to be called train

print(f"Raw dataset size: {len(raw_dataset)}")

/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Raw dataset size: 8179


In [4]:
split_dataset = raw_dataset.train_test_split(test_size=0.1)
print(f"Train dataset size: {len(split_dataset['train'])}")
print(f"Test dataset size: {len(split_dataset['test'])}")
split_dataset['train'][0]

Train dataset size: 7361
Test dataset size: 818


{'conv_index': 247,
 'helper_index': 7,
 'input': ["Seeker: No I'm not having a good time at all. I am so mad at my wife for cheating and not giving our marriage a second chance.",
  'Helper: Your wife has been unfaithful to you? I can understand why you are so angry about that!. Have the two of you considered counselling or are things beyond redemption now?',
  "Seeker: Yes she has been unfaithful. I think this person she is with is living with her now or staying there (I'm back in my home state now)",
  'Helper: I am really sorry to hear that. I do know personally how betrayed you feel when you are cheated on. All I can say is, it does get better.',
  "Seeker: I would love to try counseling. She won't even talk to me. I came back home to stay with my family for a few weeks. She called and said it's all over. I found out she has someone else. I have been miserable.",
  'Helper: It sounds like you have been treated very poorly, do you agree with that?',
  "Seeker: Yes I have and everyo

In [5]:
split_dataset['train'][0]['input'][-1]

'Helper: Hearing that she might be using drugs must be really tough for you. How are you handling this?'

In [6]:
split_dataset['train'][0]['input'][-3:-1]

['Helper: She does not seem to have stayed true to her marriage vows for very long!',
 'Seeker: No and it shocks me because she was so against not getting a divorce. We think (her bff and sister) that she may be using drugs.']

## Exploring Hyperparameter Sweeps with WanDB
Link: https://wandb.ai/matt24/vit-snacks-sweeps/reports/Hyperparameter-Search-for-HuggingFace-Transformer-Models--VmlldzoyMTUxNTg0

In [7]:
import wandb
wandb.login()


%env WANDB_PROJECT=ModernBert_SkillClassifier
# false, checkpoint 
%env WANDB_LOG_MODEL=false

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: ryanlouie2021 (ryanlouie2021-stanford-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


env: WANDB_PROJECT=ModernBert_SkillClassifier
env: WANDB_LOG_MODEL=false


In [8]:
# method
# https://wandb.ai/wandb_fc/articles/reports/What-Is-Bayesian-Hyperparameter-Optimization-With-Tutorial---Vmlldzo1NDQyNzcw
sweep_config = {
    'method': 'bayes',
    'metric': {
         'name': 'f1',
         'goal': 'maximize'  
    }
}

# hyperparameters
parameters_dict = {
    'epochs': {
        'values': [4, 10] # we can always select 
        },
    'batch_size': {
        'values': [8, 16, 32, 64] # 128 wont fit into 24GB GPU memory
        },
    # 'learning_rate': {
    #     'distribution': 'log_uniform_values',
    #     'min': 1e-5,
    #     'max': 1e-4
    # },
    'warmup_ratio': {
        'value': 0.1 # following 10% and then a linear decay; https://openreview.net/pdf?id=nzpLWnVAyah
    },
    'learning_rate': {
        'values': [1e-5, 3e-5, 5e-5, 8e-5] # these values were taken from the ModernBERT hyperparameter sweep of GLUE
    },
    'weight_decay': {
        'values': [1e-6, 5e-6, 8e-6, 1e-5] # these values were taken from the ModernBERT hyperparameter sweep of GLUE
        # 'value': 0.0
    },
    'beta': {    
        'values': [0.3, 0.6, 0.9, 0.99] # Since Beta ranges from [0, 1), we select several parameters that were found to be best in the original paper, also, selecting a few that don't use class balanced loss as much.
    },
    'context_size': {
        'values': [1, 5, 10] # some skills, like Reflections, depend wholy on what was said previously, making it easier to learn. 10 was chosen arbitrarily, 
    }
}

sweep_config['parameters'] = parameters_dict


In [9]:
from transformers import AutoModelForSequenceClassification 
from transformers import DataCollatorWithPadding
from transformers import AutoTokenizer
from transformers import Trainer, TrainingArguments
import torch

import evaluate
import numpy as np

def compute_metrics_fn(eval_preds):
    metrics = dict()
    
    accuracy_metric = evaluate.load('accuracy')
    precision_metric = evaluate.load('precision')
    recall_metric = evaluate.load('recall')
    f1_metric = evaluate.load('f1')
    
    logits = eval_preds.predictions
    labels = eval_preds.label_ids
    preds = np.argmax(logits, axis=-1)  
    
    metrics.update(accuracy_metric.compute(predictions=preds, references=labels))
    metrics.update(precision_metric.compute(predictions=preds, references=labels, average='binary'))
    metrics.update(recall_metric.compute(predictions=preds, references=labels, average='binary'))
    metrics.update(f1_metric.compute(predictions=preds, references=labels, average='binary'))

    # Print some predictions
    print(f"Some predictions: {preds[:10]}")
    
    return metrics


def get_class_weight(beta, n):
    """
    Compute class-balanced weight:
    alpha = (1 - beta) / (1 - beta^n) where n is the number of samples for the class
    
    Args:
        beta: Hyperparameter for class-balanced loss (typically between 0.9 and 0.999)
        n: Number of samples for a particular class
    
    Returns:
        The weight for the class
    """
    return (1 - beta) / (1 - beta**n)


def compute_class_balanced_loss(outputs, labels, num_items_in_batch, class_balanced_loss):
    """
    Compute class-balanced loss using the provided configuration
    
    Args:
        outputs: Model outputs containing 'logits'
        labels: Ground truth labels
        class_balanced_loss: Dictionary containing 'beta', 'n_0', and 'n_1' parameters
            - beta: Hyperparameter for class-balanced loss
            - n_0: Number of samples for class 0
            - n_1: Number of samples for class 1
    
    Returns:
        Computed loss value
    """
    logits = outputs['logits']
    
    # Compute class weights using the provided beta and class sample counts
    weights = torch.tensor([
        get_class_weight(class_balanced_loss['beta'], class_balanced_loss['n_0']),
        get_class_weight(class_balanced_loss['beta'], class_balanced_loss['n_1'])
    ])
    
    # Normalize weights
    weights = weights / weights.sum()
    
    # Move weights to the same device as logits
    weights = weights.to(device=logits.device)
    
    # Create loss function with computed weights
    criterion = torch.nn.CrossEntropyLoss(weight=weights)
    
    # Compute loss
    loss = criterion(logits, labels)
    
    return loss

def prepare_input_text(example, context_size=5):
    """
    [-6] Seeker: 
    [-5] Helper:
    [-4] Seeker: 
    [-3] Helper:
    [-2] Seeker: 
    [-1] Helper: Response to classify
    """
    # Convert the last two items of input list to a single text
    response_to_classify = example['input'][-1]
    if context_size is None:
        context = "\n".join(example['input'][:-1])
    else:
        context_start_idx = -1 - context_size
        context = "\n".join(example['input'][:-1])
    return {
        'text': f"{context}[SEP]{response_to_classify}",
        **{k:v for k,v in example.items() if k != 'input'}  # Keep other fields
    }

# def prepare_tokenized_binary_classification_dataset(dataset, which_class):
#     """
#     e.g., which_dataset = "Question-goodareas"
#     """
#     # Apply the preprocessing
#     dataset = dataset.map(prepare_input_text)
#     print(dataset['train'][0])
    
#     SKILL_OPTIONS = ["Reflections", "Validation", "Empathy", "Questions", "Suggestions", "Self-disclosure", "Structure", "Professionalism"]
#     goodareas_to_ignore = [f"{skill}-goodareas" for skill in SKILL_OPTIONS if f"{skill}-goodareas" != which_class]
#     badareas_to_ignore = [f"{skill}-badareas" for skill in SKILL_OPTIONS if f"{skill}-badareas" != which_class]
#     cols_to_remove = ['conv_index', 'helper_index', 'input', 'text']
#     cols_to_remove.extend(goodareas_to_ignore)
#     cols_to_remove.extend(badareas_to_ignore)
#     if which_class in dataset["train"].features.keys():
#         dataset =  dataset.rename_column(which_class, "labels") # to match Trainer
#     tokenized_dataset = dataset.map(lambda batch: tokenizer(batch['text'], truncation=True), batched=True, remove_columns=cols_to_remove)
#     return tokenized_dataset
    
def cleanup(things_to_delete: list | None = None):
    if things_to_delete is not None:
        for thing in things_to_delete:
            if thing is not None:
                del thing

    gc.collect()
    torch.cuda.empty_cache()
    
def train_model(config, dataset, which_class):

    # Model id to load the tokenizer
    model_id = "answerdotai/ModernBERT-large"
    # model_id = "answerdotai/ModernBERT-base"
    
    # Load Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    with wandb.init(config=config):
        # set sweep configuration
        config = wandb.config

        def prepare_input_text_fn(example):
            return prepare_input_text(example, context_size=config.context_size)
        
        dataset = dataset.map(prepare_input_text_fn)
        print(dataset['train'][0])
        
        SKILL_OPTIONS = ["Reflections", "Validation", "Empathy", "Questions", "Suggestions", "Self-disclosure", "Structure", "Professionalism"]
        goodareas_to_ignore = [f"{skill}-goodareas" for skill in SKILL_OPTIONS if f"{skill}-goodareas" != which_class]
        badareas_to_ignore = [f"{skill}-badareas" for skill in SKILL_OPTIONS if f"{skill}-badareas" != which_class]
        cols_to_remove = ['conv_index', 'helper_index', 'input', 'text']
        cols_to_remove.extend(goodareas_to_ignore)
        cols_to_remove.extend(badareas_to_ignore)
        if which_class in dataset["train"].features.keys():
            dataset = dataset.rename_column(which_class, "labels") # to match Trainer
        tokenized_dataset = dataset.map(lambda batch: tokenizer(batch['text'], truncation=True), batched=True, remove_columns=cols_to_remove)
    
        n_1 = sum(tokenized_dataset['train']['labels']) # count number of 1s
        n_0 = len(tokenized_dataset['train']['labels']) - n_1 # remaining
        print(f"Number of 1s: {n_1}, Number of 0s: {n_0}")
    
    
        # Prepare model labels - useful for inference
        labels = ["not selected", "selected"]
        num_labels = len(labels)
        label2id, id2label = dict(), dict()
        for i, label in enumerate(labels):
            label2id[label] = str(i)
            id2label[str(i)] = label
         
        # Download the model from huggingface.co/models
        model = AutoModelForSequenceClassification.from_pretrained(
            model_id, num_labels=num_labels, label2id=label2id, id2label=id2label
        )
        model.to('cuda')
        
        # Define training args
        training_args = TrainingArguments(
            output_dir= f"ModernBERT-{which_class}-classifier-sweeps",
            per_device_train_batch_size=config.batch_size,
            per_device_eval_batch_size=16,
            learning_rate=config.learning_rate,
            warmup_ratio=config.warmup_ratio, 
            num_train_epochs=config.epochs,
            weight_decay=config.weight_decay,
            bf16=True, # bfloat16 training 
            optim="adamw_torch_fused", # improved optimizer 
            # logging & evaluation strategies
            logging_strategy="epoch",
            logging_steps=100,
            eval_strategy="epoch",
            save_strategy="no", # epoch, no
            # save_total_limit=1, # needs to be commented out if save_strategy=no
            # load_best_model_at_end=True, # needs to be commented out if save_strategy=no
            # use_mps_device=True, # mps device is a mac thing
            # push to hub parameters
            report_to="wandb",
            # push_to_hub=True,
            # hub_strategy="every_save",
            # hub_token=HfFolder.get_token(),
        )


        def compute_class_balanced_loss_fn(outputs, labels, num_items_in_batch):
            return compute_class_balanced_loss(outputs, labels, num_items_in_batch, {
                    'beta': config.beta,
                    # 'beta': 0.99,
                    'n_0': n_0,
                    'n_1': n_1
                })
        
        hf_data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
        
        
        # Create a Trainer instance
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_dataset["train"],
            eval_dataset=tokenized_dataset["test"],
            processing_class=tokenizer,
            data_collator=hf_data_collator,
            compute_metrics=compute_metrics_fn,
            compute_loss_func=compute_class_balanced_loss_fn, # custom class balanced loss
        )

        try:
            trainer.train()
        except:
            cleanup(things_to_delete=[model, trainer, tokenizer, tokenized_dataset, hf_data_collator])


In [ ]:
def run_sweep(which_class):
    sweep_id = wandb.sweep(sweep_config, project=f'modernbert-{which_class}-sweeps')
    def config_fn(config=None):
        return train_model(config=config, dataset=split_dataset, which_class=which_class)
    wandb.agent(sweep_id, config_fn, count=64)

# classifier_types = ['goodareas', 'badareas']
classifier_types = ['goodareas']
# SKILL_OPTIONS = ["Reflections", "Validation", "Empathy", "Questions", "Suggestions", "Self-disclosure", "Structure", "Professionalism"]
SKILL_OPTIONS = ["Reflections", "Empathy"]
for classifier_type in classifier_types:
    for skill in SKILL_OPTIONS:
        run_sweep(f"{skill}-{classifier_type}")

Create sweep with ID: rk0rdksc
Sweep URL: https://wandb.ai/ryanlouie2021-stanford-university/modernbert-Reflections-goodareas-sweeps/sweeps/rk0rdksc


wandb: Agent Starting Run: endi01g8 with config:
wandb: 	batch_size: 64
wandb: 	beta: 0.9
wandb: 	context_size: 1
wandb: 	epochs: 10
wandb: 	learning_rate: 3e-05
wandb: 	warmup_ratio: 0.1
wandb: 	weight_decay: 8e-06
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 818/818 [00:00<00:00, 2609.42 examples/s]


{'conv_index': 247, 'helper_index': 7, 'input': ["Seeker: No I'm not having a good time at all. I am so mad at my wife for cheating and not giving our marriage a second chance.", 'Helper: Your wife has been unfaithful to you? I can understand why you are so angry about that!. Have the two of you considered counselling or are things beyond redemption now?', "Seeker: Yes she has been unfaithful. I think this person she is with is living with her now or staying there (I'm back in my home state now)", 'Helper: I am really sorry to hear that. I do know personally how betrayed you feel when you are cheated on. All I can say is, it does get better.', "Seeker: I would love to try counseling. She won't even talk to me. I came back home to stay with my family for a few weeks. She called and said it's all over. I found out she has someone else. I have been miserable.", 'Helper: It sounds like you have been treated very poorly, do you agree with that?', "Seeker: Yes I have and everyone says to mov

Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 818/818 [00:00<00:00, 2408.24 examples/s]


...

Number of 1s: 753, Number of 0s: 6608


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
wandb: WARNING Config item 'learning_rate' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'weight_decay' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'warmup_ratio' was locked by 'sweep' (ignored update).


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.412500,0.349614,0.904645,0.000000,0.000000,0.000000


Some predictions: [0 0 0 0 0 0 0 0 0 0]


/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Using the model to make predictions

In [186]:
import pandas as pd


condition = "control"
# condition = "treatment"
# input_data = pd.read_csv(f"../Empathy-Mental-Health/dataset/all_{condition}_seekerhelper_pairs.csv")
input_data = pd.read_csv("N94_all_seekerhelper_pairs.csv")
print(len(input_data))
input_data.head()

3842


,id,seeker_post,response_post
0,14_a1df3c7155d0438b9c4084b57b66c6e6_0_0,NaN,good evening I understand you're feeling isola...
1,14_a1df3c7155d0438b9c4084b57b66c6e6_0_1,"Yeah, it's just... everyone was with their fam...",I hear you. It can be difficult growing apart ...
2,14_a1df3c7155d0438b9c4084b57b66c6e6_0_2,"Yeah, it's tough. You know, during the holiday...",Perhaps they're feeling similarly. Waiting for...
3,14_a1df3c7155d0438b9c4084b57b66c6e6_0_3,But why should I always have to be the bigger ...,"Being the bigger person can feel burdensome, c..."
4,14_a1df3c7155d0438b9c4084b57b66c6e6_0_4,"Yeah, exactly. It's like, why should I keep pu...",How would you like for them to show you they c...


In [187]:
from transformers import pipeline
 
# load model from huggingface.co/models using our repository id
classifier = pipeline("sentiment-analysis", model=f"ModernBERT-{which_class}-classifier", device=0)
# classifier = pipeline("sentiment-analysis", model="ModernBERT-Empathy-goodareas-classifier", device=0)

# sample = f"Seeker: {input_data.loc[0, "seeker_post"]}\nHelper: {input_data.loc[0, "response_post"]}"
# pred = classifier(sample)
# print(pred)

def binary_prediction_seeker_response_post(seeker, helper):
    sample = f"Seeker: {seeker}\nHelper: {helper}"
    pred = classifier(sample)
    return int(pred[0]['label'] == 'selected')

Device set to use cuda:0


In [188]:
# output_preds = input_data.apply(binary_prediction_seeker_response_post, axis=0)

strengths = [binary_prediction_seeker_response_post(input_data.loc[i, "seeker_post"], input_data.loc[i, "response_post"])
             for i in range(len(input_data))]

In [189]:
input_data[f"{which_class}"] = strengths
# input_data["Empathy-goodareas"] = strengths

In [193]:
print(len(input_data))
input_data.head()

3842


,id,seeker_post,response_post,Reflections-goodareas
0,14_a1df3c7155d0438b9c4084b57b66c6e6_0_0,NaN,good evening I understand you're feeling isola...,0
1,14_a1df3c7155d0438b9c4084b57b66c6e6_0_1,"Yeah, it's just... everyone was with their fam...",I hear you. It can be difficult growing apart ...,1
2,14_a1df3c7155d0438b9c4084b57b66c6e6_0_2,"Yeah, it's tough. You know, during the holiday...",Perhaps they're feeling similarly. Waiting for...,0
3,14_a1df3c7155d0438b9c4084b57b66c6e6_0_3,But why should I always have to be the bigger ...,"Being the bigger person can feel burdensome, c...",0
4,14_a1df3c7155d0438b9c4084b57b66c6e6_0_4,"Yeah, exactly. It's like, why should I keep pu...",How would you like for them to show you they c...,0


In [195]:
# input_data.to_csv(f'all_{condition}_seekerhelper_pairs_{which_class}.csv')
input_data.to_csv(f"N94_all_seekerhelper_pairs_{which_class}.csv")

In [192]:
f'all_{condition}_seekerhelper_pairs_{which_class}.csv'

'all_control_seekerhelper_pairs_Reflections-goodareas.csv'